In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
dbutils.fs.mkdirs("/Volumes/ecommerce/raw/raw_data/streaming/orders")

True

In [0]:
BOOTSTRAP_SERVER = 'pkc-619z3.us-east1.gcp.confluent.cloud:9092'

API_KEY = 'OYJX3Z4OVKDUCZ2W'
API_SECRET = 'cfltJiNGy6H8BmFHJTodSM7ZR/pBeJO/e0pA0DxK56PQEWwxlwZc/d4oGO6Vzrbw'

In [0]:
orders_raw = (
    spark.readStream\
        .format("kafka")\
        .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER)\
            .option("subscribe", "ORDER_TOPICS")
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option(
            "kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
        )
        .option("startingOffsets", "earliest")
        .load()
)

In [0]:
display(
    orders_raw.selectExpr(
        "CAST(key AS STRING) AS key",
        "CAST(value AS STRING) AS value",
        "topic",
        "partition",
        "offset",
        "timestamp"
    ),
    checkpointLocation="/Volumes/ecommerce/raw/raw_data/checkpoints/kafka_orders_test"
)

Checkpointing to /Volumes/ecommerce/raw/raw_data/checkpoints/kafka_orders_test


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8264715193464888>, line 1
----> 1 display(
      2     orders_raw.selectExpr(
      3         "CAST(key AS STRING) AS key",
      4         "CAST(value AS STRING) AS value",
      5         "topic",
      6         "partition",
      7         "offset",
      8         "timestamp"
      9     ),
     10     checkpointLocation="/Volumes/ecommerce/raw/raw_data/checkpoints/kafka_orders_test"
     11 )

File <command-8264715193464886>, line 2
      1 orders_raw = (
----> 2     spark.readStream\
      3         .format("kafka")\
      4         .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER)\
      5             .option("subscribe", "ORDER_TOPICS")
      6         .option("kafka.security.protocol", "SASL_SSL")
      7         .option("kafka.sasl.mechanism", "PLAIN")
      8         .option(
      9             "kafka.sasl.

In [0]:
# DEfining schema

order_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("operation", StringType(), True)
])

In [0]:
orders_parsed = (
    orders_raw
    .selectExpr("CAST(value AS STRING) AS json_value")
    .select(
        F.from_json("json_value", order_schema).alias("data")
    )
    .select("data.*")
    .withColumn(
        "event_ts",
        F.to_timestamp("event_ts")
    )
)

In [0]:
bronze_query = (
    orders_parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/ecommerce/raw/raw_data/checkpoints/brz_order"
    )
    .trigger(availableNow=True)
    .toTable("ecommerce.raw.brz_orders_cdc")
)

In [0]:
%sql
DESCRIBE TABLE ecommerce.raw.slv_orders;

col_name,data_type,comment
order_id,string,null
customer_id,string,null
product_id,string,null
order_date,date,null
qty,int,null
unit_price,double,null
payment_method,string,null
order_status,string,null


In [0]:
orders_parsed.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- operation: string (nullable = true)



In [0]:
%sql
SELECT *
FROM ecommerce.raw.brz_orders_cdc
ORDER BY event_ts DESC
LIMIT 20;

event_id,event_ts,order_id,customer_id,product_id,quantity,unit_price,status,operation
bd650f55-d503-4f1a-aabc-7cdc7324cd5b,2026-09-05T20:49:48.684Z,ORD999999,CUST00001,PROD00001,2,1500.0,Delivered,U
803677ee-9ad8-4241-9770-124838b27210,2026-09-05T20:49:42.158Z,ORD999999,CUST00001,PROD00001,2,1500.0,Shipped,I
1755b391-99da-4fdb-8d1e-f188c8affad5,2026-09-05T20:45:58.363Z,ORD999999,CUST00001,PROD00001,2,1500.0,Delivered,U
ab494efe-9561-4df1-bcbd-6e1fde9e265a,2026-09-05T20:45:50.714Z,ORD999999,CUST00001,PROD00001,2,1500.0,Shipped,I
ec436012-c879-40fe-a45b-6a0796464854,2026-09-05T11:11:37.471Z,ORD90001,CUST00001,PROD00001,2,1500.0,Delivered,U
267cd95f-3a63-4c08-8c4b-3ce9ae25bfce,2026-09-05T11:11:30.978Z,ORD90001,CUST00001,PROD00001,2,1500.0,Shipped,I
058c8543-703c-4e53-aa57-ab30003c6ee0,2026-09-05T11:06:21.541Z,ORD35799,CUST00001,PROD00003,1,3500.0,Shipped,I
9c9fe70c-0c5c-4ff1-8195-b2ffdf3e1fd7,2026-09-05T11:06:19.270Z,ORD27074,CUST00003,PROD00005,5,5500.0,Delivered,I
304ffcfc-cc3e-4ce6-8515-dcee20569b22,2026-09-05T11:06:16.999Z,ORD77004,CUST00001,PROD00001,5,1500.0,Cancelled,I
58ea17fc-c275-4e42-ad97-f708010cfe6c,2026-09-05T11:06:14.726Z,ORD51588,CUST00005,PROD00002,5,2500.0,Delivered,U


In [0]:
%sql
select count(*) from ecommerce.raw.brz_orders_cdc;

count(*)
1292


In [0]:
%sql
SELECT
    order_id,
    status,
    operation,
    event_ts
FROM ecommerce.raw.brz_orders_cdc
WHERE order_id = 'ORD999999'
ORDER BY event_ts;

order_id,status,operation,event_ts
ORD999999,Shipped,I,2026-09-05T20:45:50.714Z
ORD999999,Delivered,U,2026-09-05T20:45:58.363Z
ORD999999,Shipped,I,2026-09-05T20:49:42.158Z
ORD999999,Delivered,U,2026-09-05T20:49:48.684Z


### Silver table

#### Reading Brz table

In [0]:
# Reading streaming Bronze table

streaming_brz = spark.readStream.table('ecommerce.raw.brz_orders_cdc')

#### Tranforming into silver format

In [0]:
DESCRIBE TABLE ecommerce.raw.slv_orders;

  File <command-5895414521788251>, line 1
    DESCRIBE TABLE ecommerce.raw.slv_orders;
             ^
SyntaxError: invalid syntax


In [0]:
streaming_slv = (
                streaming_brz
                    .withColumn("order_date", F.to_date("event_ts"))
    .withColumn("qty", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("order_status", F.col("status"))
    .withColumn("payment_method", F.lit(None).cast("string"))
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .select(
        "event_id",          # keep this
        "event_ts",          # keep this too
        "order_id",
        "customer_id",
        "product_id",
        "order_date",
        "qty",
        "unit_price",
        "payment_method",
        "order_status"
    )
)

#### Writing to streaming silver table

In [0]:
streaming_silver_query  = (streaming_slv.writeStream
            .format('delta')
                .outputMode('append')
                .option('checkpointLocation',
                        "/Volumes/ecommerce/raw/raw_data/checkpoints/slv_order_streaming")
                .trigger(availableNow = True)
                .toTable('ecommerce.raw.slv_order_streaming')
)

In [0]:
%sql
SELECT *
FROM ecommerce.raw.slv_order_streaming
ORDER BY order_date DESC
LIMIT 10;

event_id,event_ts,order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status
ed3823df-62dc-4e3f-9056-5fca6ba3df59,2026-09-05T10:41:33.704Z,ORD54110,CUST00005,PROD00002,2026-09-05,4,2500.0,null,Delivered
c7743d3a-938f-469c-9f44-1d1a5abc1984,2026-09-05T10:41:46.032Z,ORD56776,CUST00002,PROD00001,2026-09-05,5,1500.0,null,Shipped
19c43ac7-1799-48a4-bb40-ba3a0afb8038,2026-09-05T10:41:50.564Z,ORD69449,CUST00003,PROD00005,2026-09-05,5,5500.0,null,Delivered
d173c5bb-b1c5-4977-80c4-e16b16339b81,2026-09-05T10:41:41.498Z,ORD28184,CUST00004,PROD00005,2026-09-05,5,5500.0,null,Shipped
ad3248d9-4f65-41b9-9e96-7a077b246abc,2026-09-05T10:41:39.235Z,ORD48023,CUST00003,PROD00001,2026-09-05,1,1500.0,null,Shipped
328f67af-63f7-4079-954b-31b2a216a7e9,2026-09-05T10:41:52.827Z,ORD80644,CUST00002,PROD00003,2026-09-05,5,3500.0,null,Delivered
39f833f6-afc1-41b8-9048-12614961011d,2026-09-05T10:41:55.092Z,ORD85690,CUST00001,PROD00003,2026-09-05,4,3500.0,null,Delivered
75a9b6d7-1264-4049-97a6-87b6d2ba799e,2026-09-05T10:41:48.297Z,ORD49242,CUST00002,PROD00002,2026-09-05,4,2500.0,null,Shipped
ecef6dba-b4b4-4d71-8386-1897f084326c,2026-09-05T10:41:43.768Z,ORD31370,CUST00005,PROD00001,2026-09-05,1,1500.0,null,Delivered
d56f4591-bb5f-4688-b0e8-75b0007cae84,2026-09-05T10:41:57.359Z,ORD10817,CUST00001,PROD00002,2026-09-05,2,2500.0,null,Cancelled


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.raw.brz_orders_cdc (
    event_id STRING,
    event_ts TIMESTAMP,
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    quantity INT,
    unit_price DOUBLE,
    status STRING,
    operation STRING
)
USING DELTA;